# Hierarchical RL — l'Option framework de Sutton, Precup & Singh

**Navigation** : [RL-05 — Q-learning](RL-05-MDP-Programmation-Dynamique-Q-Learning-Python.ipynb) | [Index](README.md) | [Version Python](RL-14-RL-Hierarchique-Options-Python.ipynb) | [RL-15](RL-15-GRPO-Comparatif-Multi-Graines-Python.ipynb)

Une option est un triplet `(I, π, β)` : ensemble d'initiation, politique primitive et condition de fin. Le haut niveau choisit une option, qui produit plusieurs pas avant sa prochaine décision. Nous comparons Q-learning plat, SMDP avec options fournies et apprentissage de politiques actives.

**Objectifs** : distinguer choix et arrivée, actualiser le retour intra-option, comparer des runs multi-graines et interpréter les limites. **Prérequis** : Q-learning tabulaire et BFS. **Durée** : 50 min. CPU, BCL ; pas de solveur externe.

## 1. FourRooms et abstraction temporelle

La grille 15 × 15 comporte deux murs à la ligne et colonne 7, percés aux portes `(3,7)`, `(11,7)`, `(7,3)`, `(7,11)`. Départ `(1,1)`, but `(13,13)` ; chaque pas coûte −1, sauf l'entrée au but qui rapporte +20. Le BFS voit la carte ; l'apprenant plat ne reçoit que l'état.

Pour une option de durée `τ`, le retour est `R = Σ γ^k r_k`. La cible est `R + γ^τ max Q(s′,o′)`, ou `R` au terminal. La valeur mise à jour est celle de **l'état du choix**, pas celle de l'arrivée.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; BCL ; CPU");

Runtime .NET 10.0.14 ; BCL ; CPU


### Environnement et algorithmes

Le déplacement vers un mur laisse l'agent en place. Les politiques BFS sont construites une fois pour chaque cible ; la sélection SMDP exclut une option déjà terminée et respecte son ensemble d'initiation. Une option est coupée par son propre plafond ou le budget restant.

In [2]:
#nullable enable
record OptionRL(int Target, Dictionary<int, int> Policy, HashSet<int>? Initiation = null);
record OptionRun(int Etat, int Tau, List<double> Rewards, string Raison, bool Terminal);
record Apprentissage(int? Premier, int Derniers100, Dictionary<string, double[]> Q, List<int> Succes, Dictionary<string, int> Arrets);
record CourbeSvg(string Content);
class FourRooms
{
    public const int Taille = 15, Mur = 7;
    public static readonly (int r, int c)[] Portes =
    {
        (3, 7),
        (11, 7),
        (7, 3),
        (7, 11)
    };
    public static readonly (int dr, int dc)[] Directions =
    {
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    };
    public (int r, int c) Start = (1, 1), Goal = (13, 13);
    public int N => Taille * Taille;

    public bool EstMur(int r, int c) => r < 0 || r >= Taille || c < 0 || c >= Taille || (r == Mur && c != 3 && c != 11) || (c == Mur && r != 3 && r != 11);
    public int Encode((int r, int c) p) => p.r * Taille + p.c;
    public (int r, int c) Decode(int s) => (s / Taille, s % Taille);
    public List<int> Ouverts() => Enumerable.Range(0, N).Where(s =>
    {
        var p = Decode(s);
        return !EstMur(p.r, p.c);
    }).ToList();
    public (int s, double r, bool fini) Step(int s, int a)
    {
        var p = Decode(s);
        var d = Directions[a];
        int r = p.r + d.dr, c = p.c + d.dc;
        if (EstMur(r, c))
            return (s, -1, false);
        int next = Encode((r, c));
        return next == Encode(Goal) ? (next, 20, true) : (next, -1, false);
    }

    public (Dictionary<int, int> policy, Dictionary<int, int> dist) BFS(int target)
    {
        var dist = new Dictionary<int, int>
        {
            {
                target,
                0
            }
        };
        var q = new Queue<int>();
        q.Enqueue(target);
        while (q.Count > 0)
        {
            int s = q.Dequeue();
            var p = Decode(s);
            foreach (var d in Directions)
            {
                int r = p.r + d.dr, c = p.c + d.dc;
                if (EstMur(r, c))
                    continue;
                int v = Encode((r, c));
                if (dist.TryAdd(v, dist[s] + 1))
                    q.Enqueue(v);
            }
        }

        var pol = new Dictionary<int, int>();
        foreach (int s in Ouverts())
        {
            if (s == target)
                continue;
            var p = Decode(s);
            int best = -1, bestd = int.MaxValue;
            for (int a = 0; a < 4; a++)
            {
                var d = Directions[a];
                int r = p.r + d.dr, c = p.c + d.dc;
                if (EstMur(r, c))
                    continue;
                int v = Encode((r, c));
                if (dist.TryGetValue(v, out int dv) && dv < bestd)
                {
                    bestd = dv;
                    best = a;
                }
            }

            if (best >= 0)
                pol[s] = best;
        }

        return (pol, dist);
    }

    public Dictionary<string, OptionRL> Options()
    {
        var result = new Dictionary<string, OptionRL>();
        for (int i = 0; i < Portes.Length; i++)
        {
            int t = Encode(Portes[i]);
            result[$"porte_{i}"] = new(t, BFS(t).policy);
        }

        int goal = Encode(Goal);
        result["but"] = new(goal, BFS(goal).policy);
        return result;
    }
}

static class RL
{
    public static double Retour(IEnumerable<double> rewards, double gamma)
    {
        double total = 0, w = 1;
        foreach (double r in rewards)
        {
            total += w * r;
            w *= gamma;
        }

        return total;
    }

    public static double Cible(IReadOnlyList<double> rewards, double gamma, double next, bool terminal = false, bool withTau = true) => Retour(rewards, gamma) + (terminal ? 0 : (withTau ? Math.Pow(gamma, rewards.Count) : gamma) * next);
    public static OptionRun Executer(FourRooms env, int s, OptionRL opt, int maxSteps = 80, int restant = 400, double gamma = .99, double penalite = 0)
    {
        var rewards = new List<double>();
        if (s == env.Encode(env.Goal))
            return new(s, 0, rewards, "terminal", true);
        if (s == opt.Target)
            return new(s, 0, rewards, "cible", false);
        int limit = Math.Min(maxSteps, restant);
        if (limit <= 0)
            return new(s, 0, rewards, restant <= 0 ? "budget" : "timeout", false);
        while (rewards.Count < limit)
        {
            if (!opt.Policy.TryGetValue(s, out int a))
                return new(s, rewards.Count, rewards, "politique_absente", false);
            var(s2, r, done) = env.Step(s, a);
            s = s2;
            rewards.Add(r);
            if (done)
                return new(s, rewards.Count, rewards, "terminal", true);
            if (s == opt.Target)
                return new(s, rewards.Count, rewards, "cible", false);
        }

        string why = restant <= maxSteps ? "budget" : "timeout";
        if (why == "timeout" && penalite != 0)
            rewards[^1] += penalite;
        return new(s, rewards.Count, rewards, why, false);
    }

    public static List<string> Disponibles(Dictionary<string, OptionRL> options, int s, Dictionary<string, HashSet<int>>? sets = null) => options.Where(kv => kv.Value.Target != s && (kv.Value.Initiation is null || kv.Value.Initiation.Contains(s)) && (sets is null || (sets.ContainsKey(kv.Key) && sets[kv.Key].Contains(s)))).Select(kv => kv.Key).ToList();
    public static Apprentissage SMDP(FourRooms env, Dictionary<string, OptionRL> options, int seed, int episodes = 1500, double alpha = .2, double gamma = .99, double epsStart = .4, double epsEnd = .05, int maxSteps = 80, int budget = 400, Dictionary<string, HashSet<int>>? sets = null, double penalite = 0, Func<FourRooms, int, OptionRL, int, int, double, double, OptionRun>? runner = null, bool withTau = true, Dictionary<string, double[]>? initial = null)
    {
        runner ??= Executer;
        var random = new Random(seed);
        var q = options.Keys.ToDictionary(k => k, k => initial is null ? new double[env.N] : (double[])initial[k].Clone());
        int? premier = null;
        int last = 0;
        var succ = new List<int>();
        var arrets = new Dictionary<string, int>();
        for (int ep = 0; ep < episodes; ep++)
        {
            double eps = epsStart + (epsEnd - epsStart) * (double)ep / episodes;
            int s = env.Encode(env.Start), pas = 0;
            bool done = false;
            while (!done && pas < budget)
            {
                var disponible = Disponibles(options, s, sets);
                if (disponible.Count == 0)
                {
                    arrets["aucune_option"] = arrets.GetValueOrDefault("aucune_option") + 1;
                    break;
                }

                string name = random.NextDouble() < eps ? disponible[random.Next(disponible.Count)] : disponible.OrderByDescending(k => q[k][s]).First();
                int choisi = s;
                var r = runner(env, s, options[name], maxSteps, budget - pas, gamma, penalite);
                arrets[r.Raison] = arrets.GetValueOrDefault(r.Raison) + 1;
                if (r.Tau != r.Rewards.Count || r.Tau > Math.Min(maxSteps, budget - pas))
                    throw new Exception("transition option incoherente");
                if (r.Tau == 0)
                    break;
                s = r.Etat;
                done = r.Terminal;
                pas += r.Tau;
                var suivants = Disponibles(options, s, sets);
                double prochain = suivants.Count == 0 ? 0 : suivants.Max(k => q[k][s]);
                double cible = Cible(r.Rewards, gamma, prochain, done, withTau);
                q[name][choisi] += alpha * (cible - q[name][choisi]);
            }

            succ.Add(done ? 1 : 0);
            if (done && premier is null)
                premier = ep;
            if (ep >= episodes - 100 && done)
                last++;
        }

        return new(premier, last, q, succ, arrets);
    }

    public static (int? premier, int last, List<int> succes) Plat(FourRooms env, int seed, int episodes = 3000, double alpha = .2, double gamma = .99, double epsStart = .4, double epsEnd = .05)
    {
        var rng = new Random(seed);
        var q = new double[env.N, 4];
        int? premier = null;
        int last = 0;
        var succ = new List<int>();
        for (int ep = 0; ep < episodes; ep++)
        {
            double eps = epsStart + (epsEnd - epsStart) * (double)ep / episodes;
            int s = env.Encode(env.Start), steps = 0;
            bool done = false;
            while (!done && steps < 400)
            {
                int a;
                if (rng.NextDouble() < eps)
                    a = rng.Next(4);
                else
                {
                    a = 0;
                    for (int i = 1; i < 4; i++)
                        if (q[s, i] > q[s, a])
                            a = i;
                }

                var(next, reward, end) = env.Step(s, a);
                double best = Enumerable.Range(0, 4).Max(i => q[next, i]);
                q[s, a] += alpha * (reward + (end ? 0 : gamma * best) - q[s, a]);
                s = next;
                done = end;
                steps++;
            }

            succ.Add(done ? 1 : 0);
            if (done && premier is null)
                premier = ep;
            if (ep >= episodes - 100 && done)
                last++;
        }

        return (premier, last, succ);
    }

    public static double? Mediane(IEnumerable<int?> source)
    {
        var a = source.Where(x => x.HasValue).Select(x => x!.Value).OrderBy(x => x).ToArray();
        return a.Length == 0 ? null : (a[(a.Length - 1) / 2] + a[a.Length / 2]) / 2.0;
    }

    public static (int succes, Dictionary<int, int> usage, Dictionary<int, int> atteintes) BasNiveau(FourRooms env, int seed, int episodes = 1500)
    {
        var targets = FourRooms.Portes.Select(env.Encode).ToArray();
        var distances = targets.ToDictionary(t => t, t => env.BFS(t).dist);
        var q = targets.ToDictionary(t => t, t => new double[env.N, 4]);
        var usage = targets.ToDictionary(t => t, _ => 0);
        var atteintes = targets.ToDictionary(t => t, _ => 0);
        int succes = 0;
        var rng = new Random(seed);
        for (int ep = 0; ep < episodes; ep++)
        {
            double eps = .5 + (.08 - .5) * (double)ep / episodes;
            int s = env.Encode(env.Start), pas = 0;
            bool done = false;
            while (!done && pas < 400)
            {
                int t = targets.OrderBy(x => distances[x].GetValueOrDefault(s, int.MaxValue)).First();
                usage[t]++;
                int a;
                if (rng.NextDouble() < eps)
                    a = rng.Next(4);
                else
                {
                    a = 0;
                    for (int i = 1; i < 4; i++)
                        if (q[t][s, i] > q[t][s, a])
                            a = i;
                }

                var(next, _, end) = env.Step(s, a);
                double reward = next == t ? 1 : -1;
                double best = Enumerable.Range(0, 4).Max(i => q[t][next, i]);
                q[t][s, a] += .2 * (reward + .99 * best - q[t][s, a]);
                s = next;
                done = end;
                pas++;
                if (s == t)
                {
                    succes++;
                    atteintes[t]++;
                    break;
                }
            }
        }

        return (succes, usage, atteintes);
    }

    public static double[] Lissage(double[] x, int w = 60)
    {
        if (w <= 0 || w > x.Length)
            throw new ArgumentOutOfRangeException(nameof(w));
        var result = new double[x.Length];
        for (int i = 0; i < x.Length; i++)
            for (int j = Math.Max(0, i - w / 2); j <= Math.Min(x.Length - 1, i - w / 2 + w - 1); j++)
                result[i] += x[j] / w;
        return result;
    }
}

Console.WriteLine("FourRooms, options BFS et apprenants BCL initialises");

FourRooms, options BFS et apprenants BCL initialises


### Grille et politique de porte

`#` est un mur, `S` le départ, `G` le but ; les flèches indiquent l'action de l'option vers `(7,3)` et `*` sa cible. Elles sont calculées par BFS avec la carte, non apprises.

In [3]:
var env = new FourRooms();
var options = env.Options();
var portes = FourRooms.Portes;
int depart = env.Encode(env.Start), but = env.Encode(env.Goal);
var distance = env.BFS(but).dist;
if (env.Ouverts().Count != 200 || distance[depart] != 24 || options.Count != 5 || options.Values.Any(o => o.Policy.Count != 199))
    throw new Exception("geometrie/options");
Console.WriteLine($"{env.Ouverts().Count}/225 cellules ouvertes ; BFS depart->but {distance[depart]} pas ; retour brut optimal {-23 + 20}");
for (int r = 0; r < 15; r++)
{
    var row = new StringBuilder();
    for (int c = 0; c < 15; c++)
        row.Append(env.EstMur(r, c) ? '#' : (r, c) == env.Start ? 'S' : (r, c) == env.Goal ? 'G' : '.');
    Console.WriteLine(row.ToString());
}

Console.WriteLine("Politique BFS pour la porte (7,3) :");
var optPorte = options["porte_2"];
for (int r = 0; r < 15; r++)
{
    var row = new StringBuilder();
    for (int c = 0; c < 15; c++)
    {
        int s = env.Encode((r, c));
        row.Append(env.EstMur(r, c) ? '#' : s == optPorte.Target ? '*' : optPorte.Policy.TryGetValue(s, out int a) ? "↑↓←→"[a] : '.');
    }

    Console.WriteLine(row.ToString());
}

Console.WriteLine("cinq options : " + string.Join(", ", options.Select(x => $"{x.Key}={env.Decode(x.Value.Target)} ({x.Value.Policy.Count} etats)")));

200/225 cellules ouvertes ; BFS depart->but 24 pas ; retour brut optimal -3


.......#.......


.S.....#.......


.......#.......


...............


.......#.......


... (23 outputs left out)

↑↑↑↑↑↑↑#↑↑↑↑↑↑↑


↑↑↑↑↑↑↑#↑↑↑↑↑↑↑


↑↑↑↑↑↑↑#↑↑↑↑↑↑↑


cinq options : porte_0=(3, 7) (199 etats), porte_1=(11, 7) (199 etats), porte_2=(7, 3) (199 etats), porte_3=(7, 11) (199 etats), but=(13, 13) (199 etats)


La distance BFS est 24 pas sur 200 cases ouvertes ; le meilleur retour brut est −23+20=−3. Les cinq politiques couvrent chacune 199 états : la cible elle-même termine l'option. Les flèches montrent la politique fournie vers `(7,3)`, y compris depuis les autres pièces.

## 2. Une option et ses frontières

La démonstration distingue la somme brute des récompenses et leur somme actualisée. Les contrôles portent sur le vrai apprenant et le vrai exécuteur, pas seulement une formule isolée.

In [4]:
var demo = RL.Executer(env, depart, optPorte);
double brut = demo.Rewards.Sum(), act = RL.Retour(demo.Rewards, .99);
if (demo.Tau != 8 || brut != -8 || demo.Raison != "cible" || Math.Abs(act - (-7.72553055720799)) > 1e-8)
    throw new Exception("demo");
Console.WriteLine($"porte (7,3) : arrivee {env.Decode(demo.Etat)}, tau={demo.Tau}, brut={brut:F2}, actualise={act:F5}, arret={demo.Raison}");
var fixture = new double[]
{
    2,
    4
};
double cible = RL.Cible(fixture, .5, 10), foil = RL.Cible(fixture, .5, 10, withTau: false), term = RL.Cible(fixture, .5, 10, terminal: true);
if (cible != 6.5 || foil != 9 || term != 4)
    throw new Exception("retour SMDP");
var near = new FourRooms
{
    Start = (13, 12)
};
int s0 = near.Encode(near.Start), sg = near.Encode(near.Goal);
var one = new Dictionary<string, OptionRL>
{
    {
        "autre",
        new(near.Encode((7, 3)), new Dictionary<int, int> { { s0, 3 } })
    }
};
var initial = new Dictionary<string, double[]>
{
    {
        "autre",
        new double[near.N]
    }
};
initial["autre"][sg] = 100;
var run = RL.SMDP(near, one, 0, episodes: 1, alpha: 1, epsStart: 0, epsEnd: 0, initial: initial);
if (run.Premier != 0 || run.Q["autre"][s0] != 20 || run.Q["autre"][sg] != 100)
    throw new Exception("terminal ou etat choisi");
var coupeBudget = RL.Executer(env, depart, options["but"], restant: 2);
if (coupeBudget.Tau != 2 || coupeBudget.Raison != "budget" || coupeBudget.Terminal)
    throw new Exception("budget");
var timeout = RL.Executer(env, depart, options["but"], maxSteps: 2, restant: 10, penalite: -2);
if (timeout.Raison != "timeout" || !timeout.Rewards.SequenceEqual(new[] { -1.0, -3.0 }) || Math.Abs(RL.Retour(timeout.Rewards, .5) + 2.5) > 1e-12)
    throw new Exception("timeout");
if (RL.Disponibles(new Dictionary<string, OptionRL> { { "x", new(depart, new()) } }, depart).Count != 0)
    throw new Exception("tau zero");
var vides = options.Keys.ToDictionary(k => k, _ => new HashSet<int>());
var echec = RL.SMDP(env, options, 0, episodes: 2, sets: vides);
if (echec.Premier is not null || echec.Arrets.GetValueOrDefault("aucune_option") != 2 || RL.Mediane(new int? [] { null, null })is not null || RL.Mediane(new int? [] { 0, null }) != 0)
    throw new Exception("echec ou statistique");
var restreint = new Dictionary<string, OptionRL>
{
    {
        "autre",
        one["autre"]
    },
    {
        "interdit",
        new(near.Encode((3, 7)), new Dictionary<int, int> { { s0, 3 } })
    }
};
var qi = restreint.Keys.ToDictionary(k => k, _ => new double[near.N]);
qi["interdit"][s0] = 100;
var setsTest = new Dictionary<string, HashSet<int>>
{
    {
        "autre",
        new()
        {
            s0
        }
    },
    {
        "interdit",
        new()
    }
};
var test = RL.SMDP(near, restreint, 0, episodes: 1, alpha: 1, epsStart: 0, epsEnd: 0, sets: setsTest, initial: qi);
if (test.Premier != 0 || test.Q["interdit"][s0] != 100 || test.Q["autre"][s0] != 20)
    throw new Exception("initiation restreinte");
Console.WriteLine($"oracles : SMDP={cible}, ancien retour brut=8.5, sans tau={foil}, terminal={term} ; Qbut non nul, etat choisi, budget, timeout, tau0, initiations et aucun succes : OK");

porte (7,3) : arrivee (7, 3), tau=8, brut=-8.00, actualise=-7.72553, arret=cible


oracles : SMDP=6.5, ancien retour brut=8.5, sans tau=9, terminal=4 ; Qbut non nul, etat choisi, budget, timeout, tau0, initiations et aucun succes : OK


Huit pas produisent −8 brut et environ −7,72553 actualisé à `γ=0,99`. Avec `[2,4]`, `γ=0,5` et `Q′=10`, la cible SMDP vaut 6,5 ; une somme brute donnerait 8,5, et remplacer `γ^τ` par `γ` donne 9. Au terminal, aucune valeur du but n'est propagée.

## 3. Référence plate : quatre graines

Q-learning à quatre actions primitives : 3000 épisodes par graine, `α=0,2`, `γ=0,99`, `ε` décroissant de 0,4 à 0,05, maximum 400 pas par épisode. `first_solved` est un index à partir de zéro, pas une mesure de convergence.

In [5]:
var seeds = new[]
{
    0,
    1,
    7,
    42
};
var flat = new Dictionary<int, (int? premier, int last, List<int> succes)>();
foreach (int seed in seeds)
{
    var res = RL.Plat(env, seed);
    flat[seed] = res;
    Console.WriteLine($"plat seed={seed} : premier={res.premier?.ToString() ?? "aucun"}, succes 100 derniers={res.last}");
}

Console.WriteLine($"mediane premier succes={RL.Mediane(flat.Values.Select(x => x.premier))?.ToString() ?? "absente"}");

plat seed=0 : premier=9, succes 100 derniers=100


plat seed=1 : premier=6, succes 100 derniers=100


plat seed=7 : premier=11, succes 100 derniers=100


plat seed=42 : premier=8, succes 100 derniers=100


mediane premier succes=8.5


Les colonnes rapportent le premier succès et le nombre de réussites dans les 100 derniers épisodes. La dispersion décrit ces quatre graines ; un premier succès seul ne prouve pas que la politique a convergé.

## 4. SMDP : cinq options BFS

Le niveau haut apprend des valeurs avec les politiques de navigation déjà fournies. Quatre graines identiques au plat, mais **1500 épisodes** au lieu de 3000 : les nombres d'épisodes et les coûts de calcul ne sont pas égaux. Chaque option est limitée à 80 pas et chaque épisode à 400 pas primitifs.

In [6]:
var haut = new Dictionary<int, Apprentissage>();
foreach (int seed in seeds)
{
    var res = RL.SMDP(env, options, seed);
    haut[seed] = res;
    Console.WriteLine($"options seed={seed} : premier={res.Premier?.ToString() ?? "aucun"}, succes 100 derniers={res.Derniers100}");
}

Console.WriteLine($"mediane premier succes={RL.Mediane(haut.Values.Select(x => x.Premier))?.ToString() ?? "absente"}");

options seed=0 : premier=0, succes 100 derniers=100


options seed=1 : premier=0, succes 100 derniers=100


options seed=7 : premier=0, succes 100 derniers=100


options seed=42 : premier=0, succes 100 derniers=100


mediane premier succes=0


Les résultats ci-dessus incluent une compétence BFS fournie à l'avance. Un succès au premier épisode ne démontre pas un apprentissage plus rapide ; il n'implique pas non plus que les valeurs du haut niveau restent inchangées. Les graines .NET et NumPy ne produisent pas les mêmes suites aléatoires.

### Exercice 2 — ensembles d'initiation

Pour chaque cible, retourner les états ouverts de sa pièce, cible exclue. Sur une frontière, `r=7` est affecté au côté inférieur et `c=7` au côté droit ; les portes restent praticables. Le harnais compare les cinq ensembles entiers et utilise le résultat dans le vrai learner. Des contraintes peuvent rendre le but inaccessible.

In [7]:
#nullable enable
static int Piece(FourRooms e, (int r, int c) p) => (p.r >= FourRooms.Mur ? 2 : 0) + (p.c >= FourRooms.Mur ? 1 : 0);
static Dictionary<string, HashSet<int>>? InitiationExercice(FourRooms e, Dictionary<string, OptionRL> options)
{
    // TODO : construire les cinq ensembles d'initiation.
    return null;
}

var domaines = InitiationExercice(env, options);
if (domaines is null)
    Console.WriteLine("Exercice 2 a completer");
else
{
    if (!domaines.Keys.ToHashSet().SetEquals(options.Keys))
        throw new Exception("cles initiation");
    foreach (var kv in options)
    {
        var attendu = env.Ouverts().Where(s => s != kv.Value.Target && Piece(env, env.Decode(s)) == Piece(env, env.Decode(kv.Value.Target))).ToHashSet();
        if (!domaines[kv.Key].SetEquals(attendu))
            throw new Exception("domaine initiation");
    }

    var essai = RL.SMDP(env, options, 0, episodes: 2, sets: domaines);
    Console.WriteLine($"Exercice 2 : cinq domaines verifies, deux episodes executes ; premier={essai.Premier?.ToString() ?? "aucun"}");
}

Exercice 2 a completer


## 5. Apprendre la politique de la cible active

Sur 1500 épisodes, seed 0, `α=0,2`, `γ=0,99`, `ε=0,5→0,08`, la cible la plus proche est choisie **à chaque pas** à l'aide de distances BFS. Seule sa table primitive est mise à jour, avec +1 à la cible et −1 sinon ; l'épisode s'arrête à la première cible atteinte ou à 400 pas.

In [8]:
var basNiveau = RL.BasNiveau(env, 0);
Console.WriteLine($"atteintes de sous-objectif : {basNiveau.succes}/1500 ({(double)basNiveau.succes / 1500:F3})");
foreach (int t in basNiveau.usage.Keys)
    Console.WriteLine($"cible {env.Decode(t)} : actions {basNiveau.usage[t]}, atteintes {basNiveau.atteintes[t]}");

atteintes de sous-objectif : 1466/1500 (0.977)


cible (3, 7) : actions 70787, atteintes 440


cible (11, 7) : actions 0, atteintes 0


cible (7, 3) : actions 64382, atteintes 1026


cible (7, 11) : actions 0, atteintes 0


Cette expérience utilise un oracle connaissant la carte pour choisir la cible. Les comptes par cible montrent quelles tables ont réellement été sollicitées ; l'agrégat ne prouve pas que chacune des quatre politiques a été apprise. Ce n'est pas la mise à jour hors politique générale de toutes les options compatibles.

### Exercice 3 — pénalité à l'expiration

Implémenter un exécuteur compatible avec le learner : si la limite de 80 pas de l'option est atteinte avant la cible et avant le budget, ajouter −2 à la dernière récompense, pondérée ensuite par `γ^(τ−1)`. Ce shaping est une hypothèse à évaluer, pas une garantie générale contre les cycles.

In [9]:
#nullable enable
static OptionRun? Penalisee(FourRooms e, int s, OptionRL o, int maxSteps, int restant, double gamma, double penalite)
{
    // TODO : conserver le contrat de RL.Executer et appliquer -2 seulement au timeout.
    return null;
}

int coin = env.Encode((0, 0));
var boucle = new OptionRL(but, new Dictionary<int, int> { { coin, 0 } });
var rt = Penalisee(env, coin, boucle, 80, 100, .99, -2);
if (rt is null)
    Console.WriteLine("Exercice 3 a completer");
else
{
    var near = new FourRooms();
    near.Start = (13, 12);
    int sn = near.Encode(near.Start);
    var terminal = new OptionRL(near.Encode((7, 3)), new Dictionary<int, int> { { sn, 3 } });
    var cas = new[]
    {
        (env, coin, boucle, 80, 100),
        (env, coin, boucle, 80, 3),
        (env, depart, options["but"], 80, 2),
        (env, depart, options["but"], 80, 0),
        (env, depart, options["porte_2"], 80, 400),
        (near, sn, terminal, 80, 400)
    };
    foreach (var(e, s, o, m, b)in cas)
    {
        var attendu = RL.Executer(e, s, o, m, b, .99, -2);
        var obtenu = Penalisee(e, s, o, m, b, .99, -2);
        if (obtenu is null || obtenu.Etat != attendu.Etat || obtenu.Tau != attendu.Tau || obtenu.Raison != attendu.Raison || obtenu.Terminal != attendu.Terminal || obtenu.Rewards.Any(x => !double.IsFinite(x)) || !obtenu.Rewards.SequenceEqual(attendu.Rewards))
            throw new Exception("transition incorrecte");
    }

    double attenduRetour = Enumerable.Range(0, 80).Sum(i => -Math.Pow(.99, i)) - 2 * Math.Pow(.99, 79);
    if (Math.Abs(RL.Retour(rt.Rewards, .99) - attenduRetour) > 1e-9)
        throw new Exception("actualisation penalite");
    var banc = new Dictionary<string, OptionRL>
    {
        {
            "autre",
            terminal
        }
    };
    var init = new Dictionary<string, double[]>
    {
        {
            "autre",
            new double[near.N]
        }
    };
    var reference = RL.SMDP(near, banc, 0, episodes: 2, alpha: 1, epsStart: 0, epsEnd: 0, initial: init, penalite: -2);
    var observe = RL.SMDP(near, banc, 0, episodes: 2, alpha: 1, epsStart: 0, epsEnd: 0, initial: init, penalite: -2, runner: (e, s, o, m, r, g, p) => Penalisee(e, s, o, m, r, g, p)!);
    if (reference.Premier != observe.Premier || reference.Derniers100 != observe.Derniers100 || !reference.Succes.SequenceEqual(observe.Succes) || !reference.Q["autre"].SequenceEqual(observe.Q["autre"]) || observe.Premier != 0 || observe.Q["autre"][sn] != 20)
        throw new Exception("learner incorrect");
    Console.WriteLine("Exercice 3 : transitions et learner vérifiés sur fixtures, y compris terminal et budget");
}

Exercice 3 a completer


## 6. Synthèse et courbes

Le tableau affiche médiane du premier succès, dispersion et taux de réussite sur les 100 derniers épisodes. Les options ont des politiques BFS données ; les quatre graines et les budgets ne suffisent pas à isoler un effet causal de l'abstraction.

In [10]:
foreach (var(nom, premiers, lasts)in new[]
{
    ("plat", flat.Values.Select(x => x.premier).ToArray(), flat.Values.Select(x => x.last).ToArray()),
    ("options", haut.Values.Select(x => x.Premier).ToArray(), haut.Values.Select(x => x.Derniers100).ToArray())
}

)
{
    var ok = premiers.Where(x => x.HasValue).Select(x => x!.Value).ToArray();
    Console.WriteLine($"{nom} : mediane={RL.Mediane(premiers)?.ToString() ?? "absente"}, min/max={(ok.Length == 0 ? "absents" : $"{ok.Min()}/{ok.Max()}")}, succes moyen 100 derniers={lasts.Average() / 100:F2}");
}

plat : mediane=8.5, min/max=6/11, succes moyen 100 derniers=1.00


options : mediane=0, min/max=0/0, succes moyen 100 derniers=1.00


La courbe suivante est une expérience distincte : 600 épisodes, graines 0, 1 et 7, `ε=0,2` fixe. On prend la moyenne inter-graines, puis une convolution largeur 60 en mode `same`, avec zéros aux bords. Un dernier point proche de 31/60 peut correspondre à 100 % de succès brut en fin de série.

In [11]:
if (!RL.Lissage(new double[] { 1, 2, 3, 4 }, 2).SequenceEqual(new double[] { .5, 1.5, 2.5, 3.5 }))
    throw new Exception("lissage largeur 2");
var cs = new[]
{
    0,
    1,
    7
};
int episodesCourbe = 600;
var brutPlat = new double[episodesCourbe];
var brutOptions = new double[episodesCourbe];
foreach (int seed in cs)
{
    var p = RL.Plat(env, seed, episodes: episodesCourbe, epsStart: .2, epsEnd: .2).succes;
    var o = RL.SMDP(env, options, seed, episodes: episodesCourbe, epsStart: .2, epsEnd: .2).Succes;
    for (int i = 0; i < episodesCourbe; i++)
    {
        brutPlat[i] += p[i] / 3.0;
        brutOptions[i] += o[i] / 3.0;
    }
}

var courbePlat = RL.Lissage(brutPlat);
var courbeOptions = RL.Lissage(brutOptions);
var ones = RL.Lissage(Enumerable.Repeat(1.0, 600).ToArray());
if (Math.Abs(ones[^1] - 31.0 / 60) > 1e-12)
    throw new Exception("padding convolution");
Formatter.Register<CourbeSvg>((v, w) => w.Write(v.Content), "text/html");
var svg = new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='800' height='500' viewBox='0 0 800 500'><rect width='800' height='500' fill='white'/><text x='400' y='30' text-anchor='middle' font-size='19'>Succès par épisode — moyenne de 3 graines</text><line x1='80' y1='400' x2='740' y2='400' stroke='black'/><line x1='80' y1='60' x2='80' y2='400' stroke='black'/>");
for (int i = 0; i <= 6; i++)
{
    int x = 80 + i * 110;
    svg.Append($"<line x1='{x}' y1='400' x2='{x}' y2='405' stroke='black'/><text x='{x}' y='422' text-anchor='middle' font-size='12'>{i * 100}</text>");
}

for (int i = 0; i <= 5; i++)
{
    int y = 400 - i * 68;
    svg.Append($"<line x1='75' y1='{y}' x2='80' y2='{y}' stroke='black'/><text x='68' y='{y + 4}' text-anchor='end' font-size='12'>{i * .2:F1}</text>");
}

string Poly(double[] a) => string.Join(" ", a.Select((v, i) => $"{80 + 660.0 * i / 599:F2},{400 - 340 * v:F2}"));
svg.Append($"<polyline fill='none' stroke='#b33' stroke-width='2' points='{Poly(courbePlat)}'/><polyline fill='none' stroke='#1473a8' stroke-width='2' points='{Poly(courbeOptions)}'/>");
svg.Append("<text x='410' y='458' text-anchor='middle'>Épisode</text><text x='20' y='225' transform='rotate(-90 20 225)' text-anchor='middle'>Succès lissé</text><text x='160' y='480' fill='#b33'>— Q-learning plat</text><text x='390' y='480' fill='#1473a8'>— SMDP options</text></svg>");
display(new CourbeSvg(svg.ToString()));
Console.WriteLine($"derniers points lisses : plat={courbePlat[^1]:F3}, options={courbeOptions[^1]:F3} ; padding sur 60, exemple brut constant=1 donne {ones[^1]:F3}");

Succès par épisode — moyenne de 3 graines 0 100 200 300 400 500 600 0.0 0.2 0.4 0.6 0.8 1.0 <polyline fill='none' stroke='#b33' stroke-width='2' points='80.00,333.89 81.10,330.11 82.20,324.44 83.31,318.78 84.41,313.11 85.51,307.44 86.61,305.56 87.71,299.89 88.81,298.00 89.92,292.33 91.02,286.67 92.12,281.00 93.22,277.22 94.32,271.56 95.43,265.89 96.53,260.22 97.63,254.56 98.73,248.89 99.83,243.22 100.93,237.56 102.04,231.89 103.14,226.22 104.24,220.56 105.34,214.89 106.44,209.22 107.55,203.56 108.65,197.89 109.75,192.22 110.85,186.56 111.95,180.89 113.06,175.22 114.16,169.56 115.26,163.89 116.36,158.22 117.46,152.56 118.56,146.89 119.67,141.22 120.77,135.56 121.87,131.78 122.97,128.00 124.07,124.22 125.18,122.33 126.28,118.56 127.38,114.78 128.48,109.11 129.58,107.22 130.68,101.56 131.79,97.78 132.89,95.89 133.99,92.11 135.09,90.22 136.19,88.33 137.30,84.56 138.40,80.78 139.50,78.89 140.60,77.00 141.70,77.00 142.80,75.11 143.91,73.22 145.01,71.33 146.11,71.33 147.21,69.44 148.31,69.44 149.42,69.44 150.52,69.44 151.62,69.44 152.72,65.67 153.82,65.67 154.92,61.89 156.03,61.89 157.13,61.89 158.23,61.89 159.33,60.00 160.43,60.00 161.54,60.00 162.64,60.00 163.74,60.00 164.84,60.00 165.94,60.00 167.05,60.00 168.15,60.00 169.25,60.00 170.35,60.00 171.45,60.00 172.55,60.00 173.66,60.00 174.76,60.00 175.86,60.00 176.96,60.00 178.06,60.00 179.17,60.00 180.27,60.00 181.37,60.00 182.47,60.00 183.57,60.00 184.67,60.00 185.78,60.00 186.88,60.00 187.98,60.00 189.08,60.00 190.18,60.00 191.29,60.00 192.39,60.00 193.49,60.00 194.59,60.00 195.69,60.00 196.79,60.00 197.90,60.00 199.00,60.00 200.10,60.00 201.20,60.00 202.30,60.00 203.41,60.00 204.51,60.00 205.61,60.00 206.71,60.00 207.81,60.00 208.91,60.00 210.02,60.00 211.12,60.00 212.22,60.00 213.32,60.00 214.42,60.00 215.53,60.00 216.63,60.00 217.73,60.00 218.83,60.00 219.93,60.00 221.04,60.00 222.14,60.00 223.24,60.00 224.34,60.00 225.44,60.00 226.54,60.00 227.65,60.00 228.75,60.00 229.85,60.00 230.95,60.00 232.05,60.00 233.16,60.00 234.26,60.00 235.36,60.00 236.46,60.00 237.56,60.00 238.66,60.00 239.77,60.00 240.87,60.00 241.97,60.00 243.07,60.00 244.17,60.00 245.28,60.00 246.38,60.00 247.48,60.00 248.58,60.00 249.68,60.00 250.78,60.00 251.89,60.00 252.99,60.00 254.09,60.00 255.19,60.00 256.29,60.00 257.40,60.00 258.50,60.00 259.60,60.00 260.70,60.00 261.80,60.00 262.90,60.00 264.01,60.00 265.11,60.00 266.21,60.00 267.31,60.00 268.41,60.00 269.52,60.00 270.62,60.00 271.72,60.00 272.82,60.00 273.92,60.00 275.03,60.00 276.13,60.00 277.23,60.00 278.33,60.00 279.43,60.00 280.53,60.00 281.64,60.00 282.74,60.00 283.84,60.00 284.94,60.00 286.04,60.00 287.15,60.00 288.25,60.00 289.35,60.00 290.45,60.00 291.55,60.00 292.65,60.00 293.76,60.00 294.86,60.00 295.96,60.00 297.06,60.00 298.16,60.00 299.27,60.00 300.37,60.00 301.47,60.00 302.57,60.00 303.67,60.00 304.77,60.00 305.88,60.00 306.98,60.00 308.08,60.00 309.18,60.00 310.28,60.00 311.39,60.00 312.49,60.00 313.59,60.00 314.69,60.00 315.79,60.00 316.89,60.00 318.00,60.00 319.10,60.00 320.20,60.00 321.30,60.00 322.40,60.00 323.51,60.00 324.61,60.00 325.71,60.00 326.81,60.00 327.91,60.00 329.02,60.00 330.12,60.00 331.22,60.00 332.32,60.00 333.42,60.00 334.52,60.00 335.63,60.00 336.73,60.00 337.83,60.00 338.93,60.00 340.03,60.00 341.14,60.00 342.24,60.00 343.34,60.00 344.44,60.00 345.54,60.00 346.64,60.00 347.75,60.00 348.85,60.00 349.95,60.00 351.05,60.00 352.15,60.00 353.26,60.00 354.36,60.00 355.46,60.00 356.56,60.00 357.66,60.00 358.76,60.00 359.87,60.00 360.97,60.00 362.07,60.00 363.17,60.00 364.27,60.00 365.38,60.00 366.48,60.00 367.58,60.00 368.68,60.00 369.78,60.00 370.88,60.00 371.99,60.00 373.09,60.00 374.19,60.00 375.29,60.00 376.39,60.00 377.50,60.00 378.60,60.00 379.70,60.00 380.80,60.00 381.90,60.00 383.01,60.00 384.11,60.00 385.21,60.00 386.31,60.00 387.41,60.00 388.51,60.00 389.62,60.00 390.72,60.00 391.82,60.00 392.92,60.00 394.02,60.00 395.13,60.00 396.23,60.00 397.33,60.00 398.43,60.00 399.53,60.00 400.63,60.00 401.74,60.00 402.84,60.

derniers points lisses : plat=0.517, options=0.517 ; padding sur 60, exemple brut constant=1 donne 0.517


Les points finaux ci-dessus sont des valeurs **lissées**, non des taux bruts. Ils tiennent compte du remplissage nul ; il faut consulter les observations brutes ou les 100 derniers épisodes pour interpréter la réussite finale.

## 7. Exemple guidé — ignorer `τ`

Variante résolue de 600 épisodes, seed 0 : elle utilise le même retour actualisé et les mêmes conditions d'arrêt, mais remplace `γ^τ` par `γ` au bootstrap. Le seul contraste est cette puissance.

In [12]:
var sansTau = RL.SMDP(env, options, 0, episodes: 600, epsStart: .2, epsEnd: .2, withTau: false);
Console.WriteLine($"sans tau : premier episode resolu={sansTau.Premier?.ToString() ?? "aucun"} ; bootstrap gamma au lieu de gamma^tau");

sans tau : premier episode resolu=0 ; bootstrap gamma au lieu de gamma^tau


### Exercice 4 — durée moyenne des cinq options

Pour chaque politique, prendre les `n` premiers états triés où elle est définie, exécuter réellement l'option et retourner la moyenne des durées. Les fixtures utilisent 3 et 20 départs ; clés exactes et valeurs finies sont vérifiées.

In [13]:
#nullable enable
static Dictionary<string, double>? MoyenneTau(FourRooms e, Dictionary<string, OptionRL> options, int n)
{
    // TODO : mesurer les durees des n premiers etats de chaque politique.
    return null;
}

var r3 = MoyenneTau(env, options, 3);
var r20 = MoyenneTau(env, options, 20);
if (r3 is null || r20 is null)
    Console.WriteLine("Exercice 4 a completer");
else
{
    foreach (var(n, r)in new[]
    {
        (3, r3),
        (20, r20)
    }

    )
    {
        if (!r.Keys.ToHashSet().SetEquals(options.Keys))
            throw new Exception("cles tau");
        foreach (var kv in options)
        {
            double attendu = kv.Value.Policy.Keys.OrderBy(x => x).Take(n).Average(s => RL.Executer(env, s, kv.Value).Tau);
            if (!double.IsFinite(r[kv.Key]) || Math.Abs(r[kv.Key] - attendu) > 1e-9)
                throw new Exception("duree tau");
        }
    }

    Console.WriteLine("Exercice 4 : cinq options, deux jeux de departs verifies");
}

Exercice 4 a completer


## Exercices — récapitulatif

L'[exercice 2](#Exercice-2-—-ensembles-d'initiation) construit les ensembles d'initiation, l'[exercice 3](#Exercice-3-—-pénalité-à-l'expiration) teste une pénalité de timeout, et l'[exercice 4](#Exercice-4-—-durée-moyenne-des-cinq-options) mesure les durées des cinq options. Les stubs restent non résolus ; l'exemple guidé sans `τ` est exécuté ci-dessus.

## À retenir

Les politiques BFS fournies connaissent la carte. Le retour intra-option est actualisé, le bootstrap tient compte de `τ` et vaut zéro au terminal. Un succès à l'épisode zéro n'est pas une preuve de convergence ; l'agrégat bas niveau ne certifie pas toutes les options.

## Vérifiez votre compréhension

<details><summary>Pourquoi actualiser les récompenses au sein d'une option ?</summary>Elles arrivent à des pas primitifs distincts : le retour vaut Σ γ^k r_k.</details>

<details><summary>Que signifie un premier succès à l'épisode zéro ?</summary>Une compétence fournie est déjà suffisante ; les valeurs du haut niveau peuvent ensuite évoluer.</details>

<details><summary>Pourquoi le dernier point lissé n'est-il pas le taux brut ?</summary>La convolution `same` remplit la fenêtre de zéros hors de la série.</details>

## Pour aller plus loin

Sutton, Precup & Singh (1999), *Between MDPs and semi-MDPs* ; Bacon, Harb & Precup (2017), *Option-Critic* ; Vezhnevets et al. (2017), *FeUdal Networks*.